# Homework 5

# Задача №1 - Можете ли вы отличить сорняки от рассады?

Теперь приступим к задаче классификации на картинках. Реализуйте программу, которая определяет тип рассады на изображении. 

Для того, чтобы определить характерные особенности каждого типа рассады, у вас есть train. Train это папка, в которой картинки уже классифицированы и лежат в соответствующих папках. Исходя из этой информации можете найти признаки, присущие конкретному растению.

Проверка вашего решения будет на происходить на test. В папке test уже нет метки класса для каждой картинки. 

[Ссылка на Яндекс-диск](https://yadi.sk/d/0Zzp0klXT0iRmA), все картинки тут.

Примеры изображений для теста:
<table><tr>
    <td> <img src="https://i.ibb.co/tbqR37m/fhj.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/6yL3Wmt/sfg.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/pvn7NvF/asd.png" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [ ]:
import cv2
import os
import numpy as np
from collections import defaultdict
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

def extract_sift_features(img_path, use_mask=True):
    """Извлекает SIFT-дескрипторы с опциональной маской зелёного цвета"""
    img = cv2.imread(img_path)    
    
    if use_mask:        
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, (35, 50, 50), (85, 255, 255))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.bitwise_and(gray, gray, mask=mask)
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    sift = cv2.SIFT_create()
    _, descriptors = sift.detectAndCompute(gray, None)
    return descriptors

def train_kmeans(train_folder, n_clusters=100, random_state=42):
    """Обучает K-Means на SIFT-дескрипторах из train"""
    all_descriptors = []
    
    for class_name in os.listdir(train_folder):
        class_path = os.path.join(train_folder, class_name)
        if not os.path.isdir(class_path):
            continue
            
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            descriptors = extract_sift_features(img_path)
            if descriptors is not None:
                all_descriptors.append(descriptors)    
    
    all_descriptors = np.vstack(all_descriptors)
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(all_descriptors)
    return kmeans

def image_to_histogram(img_path, kmeans, n_clusters):
    """Преобразует изображение в гистограмму визуальных слов"""
    descriptors = extract_sift_features(img_path)
    
    visual_words = kmeans.predict(descriptors)
    hist, _ = np.histogram(visual_words, bins=n_clusters, range=(0, n_clusters))
    return hist / hist.sum() if hist.sum() > 0 else hist


In [ ]:
train_folder = "plants/train"
test_folder = "plants/test"
output_folder = "Task1_results"
n_clusters = 100
random_state = 42

print("Обучение K-Means...")
kmeans = train_kmeans(train_folder, n_clusters, random_state)

print("Подготовка данных...")
X_train, y_train = [], []
class_names = sorted(os.listdir(train_folder))

for label, class_name in enumerate(class_names):
    class_path = os.path.join(train_folder, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        hist = image_to_histogram(img_path, kmeans, n_clusters)
        X_train.append(hist)
        y_train.append(label)    

print("Обучение классификатора...")
classifier = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=random_state))
])
classifier.fit(X_train, y_train)

print("Классификация тестовых изображений...")
results = defaultdict(list)

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)
    hist = image_to_histogram(img_path, kmeans, n_clusters)
    proba = classifier.predict_proba([hist])[0]
    pred_class = np.argmax(proba)
    results[class_names[pred_class]].append(img_name)    

print("Сохранение результатов...")
os.makedirs(output_folder, exist_ok=True)

for class_name, images in results.items():
    class_dir = os.path.join(output_folder, class_name)
    os.makedirs(class_dir, exist_ok=True)
    
    for img_name in images:
        src_path = os.path.join(test_folder, img_name)
        dst_path = os.path.join(class_dir, img_name)
        img = cv2.imread(src_path)
        if img is not None:
            cv2.imwrite(dst_path, img)

print("Результаты сохранены в папку 'Task1_results'.")

Обучение K-Means...
Подготовка данных...
Обучение классификатора...
Классификация тестовых изображений...
Сохранение результатов...
Результаты сохранены в папку 'results'.


# Задача №2 - Собери пазл (2.0).

Даны кусочки изображения, ваша задача склеить пазл в исходную картинку. 

Условия:
* Дано исходное изображение для проверки, использовать собранное изображение в самом алгоритме нельзя;
* Картинки имеют друг с другом пересечение;
* После разрезки кусочки пазлов не были повернуты или отражены;
* НЕЛЬЗЯ выбрать опорную картинку для сбора пазла, как это было в homework 3
* В процессе проверки решения пазлы могут быть перемешаны, т.е. порядок пазлов в проверке может отличаться от исходного 

Изображения расположены по [ссылке](https://disk.yandex.ru/d/XtpawH1sV9UDlg).

Примеры изображений:
<img src="puzzle/su_fighter.jpg" alt="Drawing" style="width: 300px;"/>
<table><tr>
    <td> <img src="puzzle/su_fighter_shuffle/0.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/1.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/2.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/3.jpg" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [ ]:
import os
import cv2
import numpy as np
from glob import glob
from typing import List, Optional

def save_puzzle_image(
    data: np.ndarray,
    save_path: str
) -> None:
    """Сохраняет изображение пазла без отображения."""    
        
    if data.dtype == np.float32:
        save_img = (data * 255).astype(np.uint8)
    else:
        save_img = data
        
    cv2.imwrite(save_path, cv2.cvtColor(save_img, cv2.COLOR_RGB2BGR))
    print(f"Изображение сохранено в: {save_path}")


def assemble_image_puzzle(number: str, image_paths: List[str], output_dir: str = "Task2_results") -> None:    
    """Собирает пазл и сохраняет результат без отображения."""
    os.makedirs(output_dir, exist_ok=True)    
    
    output_path = os.path.join(output_dir, f"puzzle_result_{number}.png")
    
    feature_detector = cv2.SIFT_create(
        nfeatures=1000,
        nOctaveLayers=3,
        contrastThreshold=0.04,
        edgeThreshold=8,
        sigma=1.3,
    )
    
    contrast_enhancer = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    feature_matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    
    base_image = cv2.imread(image_paths[0])    
    
    total_images = len(image_paths)
    grid_dimension = int(np.ceil(np.sqrt(total_images)))
    canvas_height = base_image.shape[0] * 2 * grid_dimension
    canvas_width = base_image.shape[1] * 2 * grid_dimension
    
    result_canvas = np.zeros((canvas_height, canvas_width, 3), dtype=np.float32)
    
    center_y = canvas_height // 2
    center_x = canvas_width // 2
    y_start = center_y - base_image.shape[0] // 2
    y_end = y_start + base_image.shape[0]
    x_start = center_x - base_image.shape[1] // 2
    x_end = x_start + base_image.shape[1]
    result_canvas[y_start:y_end, x_start:x_end] = base_image.astype(np.float32) / 255.0
    
    transformation_matrices = [[] for _ in range(total_images)]
    processing_queue = [0] 
    processed_images = set()
    candidate_images = set(range(1, total_images))
    match_ratio_threshold = 0.47

    while processing_queue:
        current_idx = processing_queue.pop()
        processed_images.add(current_idx)

        current_image = cv2.imread(image_paths[current_idx])
        if current_image is None:
            continue
        
        ycrcb_image = cv2.cvtColor(current_image, cv2.COLOR_BGR2YCrCb)
        ycrcb_image[:, :, 0] = contrast_enhancer.apply(ycrcb_image[:, :, 0])
        current_image = cv2.cvtColor(ycrcb_image, cv2.COLOR_YCrCb2BGR)

        keypoints, descriptors = feature_detector.detectAndCompute(current_image, None)
        if descriptors is None:
            continue
        
        for candidate_idx in list(candidate_images):
            candidate_image = cv2.imread(image_paths[candidate_idx])
            if candidate_image is None:
                candidate_images.remove(candidate_idx)
                continue
            
            ycrcb_candidate = cv2.cvtColor(candidate_image, cv2.COLOR_BGR2YCrCb)
            ycrcb_candidate[:, :, 0] = contrast_enhancer.apply(ycrcb_candidate[:, :, 0])
            candidate_image = cv2.cvtColor(ycrcb_candidate, cv2.COLOR_YCrCb2BGR)

            candidate_kp, candidate_desc = feature_detector.detectAndCompute(candidate_image, None)
            if candidate_desc is None:
                candidate_images.remove(candidate_idx)
                continue
            
            raw_matches = feature_matcher.knnMatch(
                descriptors.astype(np.float32), 
                candidate_desc.astype(np.float32), 
                k=2
            )
            good_matches = []
            for first_match, second_match in raw_matches:
                if first_match.distance < match_ratio_threshold * second_match.distance:
                    good_matches.append(first_match)

            if len(good_matches) >= 3:                
                current_points = np.float32([keypoints[m.queryIdx].pt for m in good_matches])
                candidate_points = np.float32([candidate_kp[m.trainIdx].pt for m in good_matches])
                
                transform_result = cv2.estimateAffine2D(candidate_points, current_points)
                transform_matrix = transform_result[0]
                if transform_matrix is not None:                    
                    processing_queue.append(candidate_idx)
                    candidate_images.remove(candidate_idx)
                    
                    transformed_image = np.zeros((canvas_height, canvas_width, 3), dtype=np.float32)
                    temp_y_start = center_y - candidate_image.shape[0] // 2
                    temp_y_end = temp_y_start + candidate_image.shape[0]
                    temp_x_start = center_x - candidate_image.shape[1] // 2
                    temp_x_end = temp_x_start + candidate_image.shape[1]
                    
                    transformed_image[temp_y_start:temp_y_end, temp_x_start:temp_x_end] = (
                        candidate_image.astype(np.float32) / 255.0
                    )
                    
                    for prev_transform in transformation_matrices[current_idx]:
                        transformed_image = cv2.warpAffine(
                            transformed_image, 
                            prev_transform, 
                            (canvas_width, canvas_height)
                        )
                    
                    transformation_matrices[candidate_idx] = (
                        transformation_matrices[current_idx] + [transform_matrix]
                    )
                    
                    transformed_image = cv2.warpAffine(
                        transformed_image, 
                        transform_matrix, 
                        (canvas_width, canvas_height)
                    )
                    
                    update_mask = (result_canvas.sum(axis=2) == 0).astype(np.float32)
                    update_mask = cv2.erode(update_mask, np.ones((3, 3)), iterations=2)
                    update_mask = cv2.dilate(update_mask, np.ones((3, 3)), iterations=2)                    
                    
                    update_mask_3d = np.repeat(update_mask[:, :, np.newaxis], 3, axis=2)
                    
                    result_canvas += update_mask_3d * transformed_image

    
    save_puzzle_image(result_canvas, output_path)

china_puzzle_paths = glob(os.path.join(os.getcwd(), "puzzle", "china_shuffle", "*"))
fighter_puzzle_paths = glob(os.path.join(os.getcwd(), "puzzle", "su_fighter_shuffle", "*"))

print("Собираем пазл 'China'...")
assemble_image_puzzle('0', china_puzzle_paths)

print("Собираем пазл 'Fighter'...")
assemble_image_puzzle('1', fighter_puzzle_paths)


Пазл 1 успешно собран
Не удалось собрать пазл 2
Не удалось собрать пазл 3
Размер result1: (220, 820, 3)
Размер result2: None
Размер result3: None
